# Capstone · Phase 4 上机：因果实验设计与验证

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据集）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 在**真实RCT数据**（NSW职业培训实验）上完成DoWhy四步因果分析（假设->识别->估计->反驳）
2. 用**DML双重机器学习**和**因果森林**估计异质因果效应（CATE）
3. 用**CUPED**利用前实验协变量降低实验方差
4. 用**自定义BaseMetric**（deepeval fallback）评估Agent输出中因果证据使用质量
5. 整合技能3（因果推断）+技能5（Agent评估），回答「营销Agent的干预真的有效吗？」

## 真实数据
- **NSW真实RCT**（`causaldata`包）：`treat`=营销干预, `re78`=转化, `re75`=基线
- **真实库**：DoWhy + econml + causaldata + 自定义BaseMetric

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install causaldata dowhy econml scikit-learn statsmodels -q

## 1. 数据集背景与营销映射

**NSW数据集**：NSW职业培训示范实验的真实数据（Dehejia & Wahba 1999），因果推断最经典的真实教学数据集。

| NSW变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat` | 是否收到营销干预（优惠券/广告/Agent系统） | 处理 T |
| `re78` | 转化率/GMV/客单价 | 结果 Y |
| `re75` | 基线消费/历史转化率 | 前实验协变量（CUPED用） |
| `age`, `educ`, `black`, `hisp`, `marr`, `nodegree`, `re74` | 用户画像特征 | 协变量 X（混杂） |

**核心因果问题**：营销干预（treat）对转化（re78）的真实因果效应（ATE）是多少？哪些用户群体效应更大（CATE）？

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import dowhy
from dowhy import CausalModel
from causaldata import nsw_mixtape
from econml.dml import LinearDML, CausalForestDML
from sklearn.ensemble import RandomForestRegressor

# 评估库（deepeval fallback）
try:
    from deepeval.metrics import BaseMetric
    HAS_DEEPEVAL = True
except ImportError:
    HAS_DEEPEVAL = False
    print('deepeval未安装，使用自定义BaseMetric fallback')

## TODO 1-2：加载真实数据 + 朴素估计

先加载数据、检查协变量均衡性，再算朴素均值差（有偏估计）。

In [ ]:
# 1. 加载真实NSW数据，检查协变量均衡性
df = nsw_mixtape.load_pandas().data
print(f'数据形状: {df.shape}')
print(f'处理组: {len(df[df["treat"]==1])}, 对照组: {len(df[df["treat"]==0])}')
print()

# 协变量均衡性检查
covariates = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
print('协变量均衡性检查：')
for col in covariates:
    t_mean = df[df['treat']==1][col].mean()
    c_mean = df[df['treat']==0][col].mean()
    print(f'  {col}: treat={t_mean:.2f}, control={c_mean:.2f}, diff={t_mean-c_mean:.2f}')

In [ ]:
# 2. 朴素估计（有偏）
naive_ate = df[df['treat']==1]['re78'].mean() - df[df['treat']==0]['re78'].mean()
print(f'朴素估计 ATE = {naive_ate:.2f}')
print('⚠️ 这个估计有偏！协变量分布不均（见上）')

## 2. 因果图（DAG）与混杂分析

NSW场景的因果结构（简化）：

```
age, educ ──> treat ──> re78
     │            ▲
     └────────────┘   (后门路径)
re74, re75 ──> treat ──> re78
     │            ▲
     └────────────┘   (后门路径)
```

- **混杂因素**：age/educ/re74/re75同时影响treat和re78
- **后门路径**：treat ← age/educ/re74/re75 → re78（创造虚假相关）
- **后门准则**：控制这些协变量即可识别treat→re78的因果效应

**营销类比**：收到优惠券的用户可能本来就是高活跃用户（自选择），朴素均值差混淆了「用户特征」和「优惠券效果」。

## TODO 3：DoWhy四步因果分析

用DoWhy完成建模->识别->估计->反驳四步因果分析。

In [ ]:
# 3. DoWhy四步因果分析
common_causes = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']

model = CausalModel(
    data=df,
    treatment='treat',
    outcome='re78',
    common_causes=common_causes
)

identified_estimand = model.identify_effect()

causal_estimate = model.estimate_effect(
    identified_estimand,
    method_name='backdoor.linear_regression'
)

refutation = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    'placebo_treatment_refuter'
)

print(f'DoWhy后门调整 ATE = {causal_estimate.value:.2f}')
print(f'安慰剂检验: {refutation}')

## 3. CUPED方差缩减

CUPED（Deng et al. 2013, Microsoft KDD）利用前实验协变量调整结果变量，缩小方差、提升检测灵敏度。

$$Y_{adj} = Y - \theta \cdot (X_{pre} - \bar{X}_{pre}), \quad \theta = \frac{\text{Cov}(Y, X_{pre})}{\text{Var}(X_{pre})}$$

在NSW中，用 `re75`（前一年收入）调整 `re78`（结果收入）。营销映射：用基线消费调整转化率。

In [ ]:
# 4. CUPED方差缩减 -- 用re75调整re78
Y = df['re78'].values
X_pre = df['re75'].values
T = df['treat'].values

theta = np.cov(Y, X_pre)[0, 1] / np.var(X_pre)
Y_adj = Y - theta * (X_pre - np.mean(X_pre))

cuped_treated = Y_adj[T == 1].mean()
cuped_control = Y_adj[T == 0].mean()
cuped_ate = cuped_treated - cuped_control
var_reduction = 1 - np.var(Y_adj) / np.var(Y)

print(f'朴素 ATE = {naive_ate:.2f}')
print(f'CUPED ATE = {cuped_ate:.2f}')
print(f'方差缩减: {var_reduction:.2%}')

## 4. DML双重机器学习

DML（Chernozhukov et al. 2018）用ML模型估计nuisance functions，再用残差化Y和T估计因果效应。相比线性回归，DML在高维协变量和非线性关系下更稳健。

用 `econml.dml.LinearDML` 估计ATE和CATE（异质因果效应）。

In [ ]:
# 5. DML双重机器学习 -- 估计ATE和CATE
covariates = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
X = df[covariates].values

dml = LinearDML(
    model_y=RandomForestRegressor(n_estimators=50, random_state=42),
    model_t=RandomForestRegressor(n_estimators=50, random_state=42),
    discrete_treatment=True,
    random_state=42
)
dml.fit(Y, T, X=X)

dml_ate = dml.ate(X=X)
dml_ci = dml.ate_interval(X=X, alpha=0.05)

# CATE: 按年龄中位数分组
age_median = np.median(df['age'])
young_mask = df['age'].values <= age_median
cate_young = dml.ate(X=X[young_mask])
cate_old = dml.ate(X=X[~young_mask])

print(f'DML ATE = {dml_ate:.2f}, 95% CI: [{dml_ci[0]:.2f}, {dml_ci[1]:.2f}]')
print(f'CATE (young, age<={age_median:.0f}): {cate_young:.2f}')
print(f'CATE (older, age>{age_median:.0f}): {cate_old:.2f}')

## 5. 因果森林

因果森林（Wager & Athey 2018）用随机森林结构估计异质因果效应（CATE），能自动发现协变量交互效应。

用 `econml.dml.CausalForestDML` 估计CATE，对比DML的异质效应发现。

In [ ]:
# 6. 因果森林 -- 估计CATE
cf = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=50, random_state=42),
    model_t=RandomForestRegressor(n_estimators=50, random_state=42),
    discrete_treatment=True,
    n_estimators=100,
    random_state=42
)
cf.fit(Y, T, X=X)

cf_ate = cf.ate(X=X)
cf_ci = cf.ate_interval(X=X, alpha=0.05)

print(f'因果森林 ATE = {cf_ate:.2f}, 95% CI: [{cf_ci[0]:.2f}, {cf_ci[1]:.2f}]')
print(f'DML ATE = {dml_ate:.2f}, 因果森林 ATE = {cf_ate:.2f}')

## 6. Agent因果证据评估（整合技能5 Day 3）

Phase 3的营销Agent系统会输出策略文本。本步评估Agent输出中**因果证据使用质量**：
- 是否引用了ATE/CATE数值？
- 是否提及混杂因素控制？
- 是否提及稳健性检验？
- 是否避免因果过度外推？

用自定义BaseMetric（deepeval fallback，无API key时规则评估）。

In [ ]:
# 7. Agent因果证据评估 -- 自定义BaseMetric
class CausalEvidenceMetric:
    """自定义因果证据评估指标（deepeval BaseMetric fallback）
    
    当deepeval不可用或无API key时，使用规则评估Agent输出中因果证据使用质量。
    评估维度：
    1. 是否引用ATE/CATE数值
    2. 是否提及混杂因素控制
    3. 是否提及稳健性检验
    4. 是否避免因果过度外推
    """
    def __init__(self, threshold: float = 0.7):
        self.threshold = threshold
        self.score = 0
        self.reason = ''
        
    def measure(self, agent_output: str):
        score = 0
        reasons = []
        text = agent_output.lower()
        
        # 维度1: 引用ATE/CATE
        has_ate = any(kw in text for kw in ['ate', '平均因果效应', 'cate', '条件平均'])
        if has_ate:
            score += 0.25
            reasons.append('引用了ATE/CATE(+0.25)')
        else:
            reasons.append('未引用ATE/CATE(-0.25)')
        
        # 维度2: 混杂因素控制
        has_confounder = any(kw in text for kw in ['混杂', 'confound', '后门', 'backdoor', '调整'])
        if has_confounder:
            score += 0.25
            reasons.append('提及混杂控制(+0.25)')
        else:
            reasons.append('未提及混杂控制(-0.25)')
        
        # 维度3: 稳健性检验
        has_robust = any(kw in text for kw in ['反驳', 'refut', '稳健', 'robust', '安慰剂', 'placebo'])
        if has_robust:
            score += 0.25
            reasons.append('提及稳健性检验(+0.25)')
        else:
            reasons.append('未提及稳健性检验(-0.25)')
        
        # 维度4: 避免过度外推
        no_overreach = not any(kw in text for kw in ['必定', '绝对', '100%', '因果证明', '确定'])
        if no_overreach:
            score += 0.25
            reasons.append('避免过度外推(+0.25)')
        else:
            reasons.append('存在过度外推(-0.25)')
        
        self.score = score
        self.reason = '; '.join(reasons)
        return score
    
    def is_successful(self):
        return self.score >= self.threshold

good_output = '''基于DoWhy因果分析，营销干预的ATE为1676（95% CI: [608, 3271]）。
我们通过后门调整控制了年龄、教育、前期收入等混杂因素。
安慰剂检验p值为0.98，新效应接近0，说明估计稳健。
建议在25岁以上用户群体中推广（CATE更高），但需注意可忽略性假设的局限。'''

bad_output = '''营销干预必定有效！转化率绝对提升了100%。
因果证明了优惠券的效果，不需要考虑混杂因素。'''

metric = CausalEvidenceMetric(threshold=0.7)
print(f'Good output score: {metric.measure(good_output):.2f}')
print(f'  Reason: {metric.reason}')
print(f'  Successful: {metric.is_successful()}')
print()
print(f'Bad output score: {metric.measure(bad_output):.2f}')
print(f'  Reason: {metric.reason}')
print(f'  Successful: {metric.is_successful()}')

## 7. 反思与前沿

### 反思问题
1. 朴素ATE vs DoWhy后门ATE vs DML ATE的差异来自哪里？哪个最可信？
2. DML和因果森林的CATE估计是否一致？哪个用户群体获益最大？
3. CUPED调整后方差降低了多少？这对实验设计有什么启示？
4. 安慰剂检验和随机混杂检验的p值是否支持你的因果估计？
5. 如果NSW数据里有个**没观测到的混杂**（如「个人上进心」），你的估计还可靠吗？

### 2026前沿
- **DML双重机器学习**：高维协变量下的无偏因果估计
- **CUPED**：方差缩减技术，等效提升样本量
- **因果森林**：自动发现异质因果效应
- **Uplift/增量建模**：按因果效应排序优先干预「可被说服」用户
- **MAB多臂老虎机**：多干预方案的自适应选择
- **贝叶斯因果推断**：小样本下的不确定性量化

### 整合性
本Phase整合了技能3（因果推断Day1-5）和技能5（Day3评估）：
- 技能3提供因果推断方法论（DoWhy四步+DML+因果森林+CUPED）
- 技能5提供Agent评估方法论（BaseMetric+LLM-as-a-judge）
- Phase 4用因果推断评估Agent系统的效果，用Agent评估验证Agent是否正确使用因果证据